Compute pilot measures of panel disagreement and final protocol quality.....


In [1]:
from pathlib import Path
import os
PROJECT_ROOT = next((p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / 'experiments/clinical_protocols').is_dir() and (p / 'data').is_dir()))
if os.name == 'nt' and (not str(PROJECT_ROOT).startswith('\\\\?\\')):
    PROJECT_ROOT = Path('\\\\?\\' + str(PROJECT_ROOT))


In [2]:
import io
import json
import os
import sys
import warnings
import numpy as np
import pandas as pd

def read_csv_normalized(*args, **kwargs):
    data = pd.read_csv(*args, **kwargs)
    if 'Vignette' in data.columns:
        data['Vignette'] = data['Vignette'].map(lambda value: value.replace('\r\n', '\n') if isinstance(value, str) else value)
    return data
import statsmodels.formula.api as smf
from scipy import stats
warnings.filterwarnings('ignore')
ROOT = str(PROJECT_ROOT)
OUT = os.path.normpath(os.path.join(ROOT, 'analyses/panel_mechanisms/pilot_disagreement_and_recovery'))
os.makedirs(OUT, exist_ok=True)
PF = 'experiments/clinical_protocols/delphi_panels/heterogeneous_gpt120_organizer_complete/llm_as_judge/delphi_base_%s_judge.csv'
B = 'experiments/clinical_protocols/single_model_baselines/baseline_%s/llm_as_judge/baseline_%s_%s_judge.csv'
RT = 'experiments/clinical_protocols/single_model_baselines/baseline_%s/llm_as_judge/judge_robustness/baseline_noroles_%s_retest_r%d_judge.csv'
MEM = ['gpt', 'qwen', 'deepseekv4']
ALL_SINGLE = ['gpt', 'qwen', 'deepseekv4', 'deepseekv4pro', 'gemini', 'glm', 'kimi']

def rd(rel):
    p = os.path.normpath(os.path.join(ROOT, rel))
    if not os.path.exists(p):
        return None
    d = read_csv_normalized(p, sep=';', encoding='utf-8-sig', low_memory=False)
    if 'table_source' in d.columns:
        d = d[d['table_source'].fillna('result_df') == 'result_df']
    return d.dropna(subset=['Judge_score', 'Vignette']).drop_duplicates('Vignette').set_index('Vignette')

def js(rel):
    d = rd(rel)
    return None if d is None else d['Judge_score'].astype(float)

def m1_emergence():

    def items(d):
        r = {}
        for v, s in d['judge_details'].dropna().items():
            try:
                arr = json.loads(s)
            except Exception:
                continue
            for it in arr:
                ax = next((t.split(':', 1)[1] for t in it.get('tags', []) if isinstance(t, str) and t.startswith('axis:')), None)
                r[v, it['id']] = (bool(it.get('met')), it.get('points', 0) or 0, ax)
        return r
    pan = items(rd(PF % 'noroles'))
    mem = {m: items(rd(B % (m, 'noroles', m))) for m in MEM}
    keys = [k for k in pan if all((k in mem[m] for m in mem)) and pan[k][1] > 0]
    rows = [{'k': sum((mem[m][k][0] for m in mem)), 'panel': pan[k][0], 'points': pan[k][1], 'axis': pan[k][2]} for k in keys]
    d = pd.DataFrame(rows)
    t = d.groupby('k').agg(пунктов=('panel', 'size'), выполнила_панель=('panel', 'mean'), средний_вес=('points', 'mean'))
    rec = d[d.k == 0]
    lost = d[d.k == 3]
    ax = rec.groupby('axis').agg(пунктов=('panel', 'size'), восстановлено=('panel', 'mean'))
    ci = {}
    for lab, s in [('восстановление из k=0', rec['panel']), ('потери при k=3', 1 - lost['panel'])]:
        k_, n_ = (int(s.sum()), len(s))
        lo, hi = stats.beta.ppf([0.025, 0.975], [k_, k_ + 1], [n_ - k_ + 1, n_ - k_])
        ci[lab] = (k_, n_, k_ / n_, lo, hi)
    return (t, ax, ci, len(keys))

def m2_min_mean_max():
    p = js(PF % 'noroles')
    M = pd.concat({m: js(B % (m, 'noroles', m)) for m in MEM}, axis=1).dropna()
    ix = p.index.intersection(M.index)
    d = pd.DataFrame({'y': p.loc[ix], 'mn': M.loc[ix].min(axis=1), 'mean': M.loc[ix].mean(axis=1), 'mx': M.loc[ix].max(axis=1)})
    rows = []
    for f in ['y ~ mn', 'y ~ mean', 'y ~ mx', 'y ~ mn + mx', 'y ~ mn + mean + mx']:
        r = smf.ols(f, d).fit()
        rec = {'спецификация': f, 'R2': r.rsquared, 'n': len(d)}
        for k in ('mn', 'mean', 'mx'):
            if k in r.params:
                rec[k] = r.params[k]
                rec['p ' + k] = r.pvalues[k]
        rows.append(rec)
    return pd.DataFrame(rows).set_index('спецификация')

def m3_weak_gain():
    p = js(PF % 'noroles')
    M = pd.concat({m: js(B % (m, 'noroles', m)) for m in MEM}, axis=1).dropna()
    ix = p.index.intersection(M.index)
    lvl = M.loc[ix].mean(axis=1)
    gain = p.loc[ix] - lvl
    naive = stats.spearmanr(lvl, gain)
    rows = [{'оценка': 'наивная (уровень из того же замера)', 'n': len(ix), 'rho': naive.statistic, 'p': naive.pvalue, 'смещение': 'регрессия к середине -> завышена по модулю'}]
    alt = []
    for m in MEM:
        r = [js(RT % (m, m, k)) for k in (1, 2)]
        r = [x for x in r if x is not None]
        if r:
            alt.append(pd.concat(r, axis=1).mean(axis=1).rename(m))
    if alt:
        A = pd.concat(alt, axis=1).dropna().mean(axis=1)
        j = ix.intersection(A.index)
        iv = stats.spearmanr(A.loc[j], gain.loc[j])
        rows.append({'оценка': 'инструментальная (уровень из retest r1-r2)', 'n': len(j), 'rho': iv.statistic, 'p': iv.pvalue, 'смещение': 'общий шум устранён'})
        r = smf.ols('g ~ lv', pd.DataFrame({'g': gain.loc[j].values, 'lv': A.loc[j].values})).fit()
        rows.append({'оценка': '  наклон при инструментальном уровне', 'n': len(j), 'rho': r.params['lv'], 'p': r.pvalues['lv'], 'смещение': 'МНК'})
    return pd.DataFrame(rows).set_index('оценка')

def m4_roles():
    rows, lev = ([], {})
    for m in ALL_SINGLE:
        a, b = (js(B % (m, 'noroles', m)), js(B % (m, 'randomroles', m)))
        if a is None or b is None:
            continue
        ix = a.index.intersection(b.index)
        d = b.loc[ix] - a.loc[ix]
        rows.append({'плечо': 'baseline ' + m, 'noroles': a.loc[ix].mean(), 'randomroles': b.loc[ix].mean(), 'Δ от ролей': d.mean(), 'p': stats.wilcoxon(d)[1]})
        lev[m] = (a.loc[ix].mean(), d.mean())
    a, b = (js(PF % 'noroles'), js(PF % 'randomroles'))
    ix = a.index.intersection(b.index)
    d = b.loc[ix] - a.loc[ix]
    rows.append({'плечо': 'P-base Delphi', 'noroles': a.loc[ix].mean(), 'randomroles': b.loc[ix].mean(), 'Δ от ролей': d.mean(), 'p': stats.wilcoxon(d)[1]})
    tab = pd.DataFrame(rows).set_index('плечо')
    r = stats.spearmanr([v[0] for v in lev.values()], [v[1] for v in lev.values()])
    recs = []
    for dsg in ['noroles', 'randomroles']:
        pp = js(PF % dsg)
        M = pd.concat({m: js(B % (m, dsg, m)) for m in MEM}, axis=1).dropna()
        i2 = pp.index.intersection(M.index)
        recs.append(pd.DataFrame({'design': dsg, 'y': pp.loc[i2].values, 'mean': M.loc[i2].mean(axis=1).values, 'rounds': pd.to_numeric(rd(PF % dsg).loc[i2, 'delphi_rounds_executed'], errors='coerce').values}))
    dd = pd.concat(recs, ignore_index=True)
    dd['gain'] = dd['y'] - dd['mean']
    per = {}
    for dsg, s in dd.groupby('design'):
        rr = smf.ols('gain ~ mean', s).fit()
        per[dsg] = (rr.params['mean'], rr.pvalues['mean'])
    inter = smf.ols("gain ~ mean * C(design, Treatment('noroles'))", dd).fit()
    ik = [i for i in inter.params.index if 'mean:' in i][0]
    by_rounds = dd.dropna(subset=['rounds']).groupby(['design', dd.dropna(subset=['rounds'])['rounds'].astype(int)]).agg(n=('y', 'size'), панель=('y', 'mean'), прирост=('gain', 'mean'))
    ri = smf.ols("gain ~ C(rounds) * C(design, Treatment('noroles')) + mean", dd.dropna(subset=['rounds'])).fit()
    rk = [i for i in ri.params.index if 'rounds' in i and 'design' in i]
    return (tab, r, per, (inter.params[ik], inter.pvalues[ik]), by_rounds, {i: (ri.params[i], ri.pvalues[i]) for i in rk})

def show(t, o, note=''):
    print('\n' + '=' * 100)
    print(t)
    if note:
        print(note)
    print('=' * 100)
    print(o.to_string() if hasattr(o, 'to_string') else o)

def main():
    print('=' * 100)
    print('ПИЛОТЫ К ГИПОТЕЗАМ О МЕХАНИЗМЕ (данные уже собраны, API не вызывается)')
    print('=' * 100)
    t, ax, ci, n = m1_emergence()
    show('M1. ЭМЕРДЖЕНТНОСТЬ НА УРОВНЕ КРИТЕРИЯ (сопоставлено %d пунктов рубрики)' % n, t, 'k -- сколько из 3 участников выполнили этот критерий поодиночке.')
    for lab, (k_, n_, pt, lo, hi) in ci.items():
        print('  %-24s %d/%d = %.3f  95%% ДИ [%.3f; %.3f]' % (lab, k_, n_, pt, lo, hi))
    show('M1b. ВОССТАНОВЛЕНИЕ ИЗ k=0 ПО ОСЯМ РУБРИКИ', ax)
    t.to_csv(os.path.normpath(os.path.join(OUT, 'criterion_recovery_by_successful_member_count.csv')), encoding='utf-8-sig')
    show('M2. ВЫХОД ПАНЕЛИ ~ МИНИМУМ / СРЕДНЕЕ / МАКСИМУМ УЧАСТНИКОВ', m2_min_mean_max(), 'Если значим только mx -- панель есть отбор лучшего; если mn тоже -- слабейший тянет вниз.')
    show('M3. «СЛАБЫМ БОЛЬШЕ»: НАИВНАЯ ОЦЕНКА ПРОТИВ ИНСТРУМЕНТАЛЬНОЙ', m3_weak_gain(), 'Инструмент снимает регрессию к середине: уровень берётся из независимых прогонов.')
    tab, rho, per, inter, by_rounds, rint = m4_roles()
    show('M4. СЛУЧАЙНЫЕ РОЛИ КАК ВТОРАЯ «ПОДДЕРЖКА»', tab)
    print('\n  роли помогают тем сильнее, чем слабее модель: rho = %+.3f, p = %.3g (по %d моделям)' % (rho.statistic, rho.pvalue, len(tab) - 1))
    print('\n  НАКЛОН «прирост ОТ ПРОТОКОЛА ~ уровень участников» по дизайнам:')
    for k, (b_, p_) in per.items():
        print('    %-12s %+.3f (p = %.2g)' % (k, b_, p_))
    print('    взаимодействие уровень x дизайн: %+.3f, p = %.3g' % inter)
    show('M4b. ТО ЖЕ ПО КОГОРТАМ ЧИСЛА РАУНДОВ', by_rounds)
    print('\n  взаимодействие раунды x дизайн: ' + ', '.join(('%s %+.3f (p=%.2g)' % (i.split('[')[1][:4], v[0], v[1]) for i, v in rint.items())))
    print('\nCSV: %s' % OUT)
if __name__ == '__main__':
    main()


ПИЛОТЫ К ГИПОТЕЗАМ О МЕХАНИЗМЕ (данные уже собраны, API не вызывается)



M1. ЭМЕРДЖЕНТНОСТЬ НА УРОВНЕ КРИТЕРИЯ (сопоставлено 2334 пунктов рубрики)
k -- сколько из 3 участников выполнили этот критерий поодиночке.
   пунктов  выполнила_панель  средний_вес
k                                        
0      643          0.399689     5.503888
1      358          0.678771     5.977654
2      366          0.855191     6.581967
3      967          0.981386     7.476732
  восстановление из k=0    257/643 = 0.400  95% ДИ [0.362; 0.439]
  потери при k=3           18/967 = 0.019  95% ДИ [0.011; 0.029]

M1b. ВОССТАНОВЛЕНИЕ ИЗ k=0 ПО ОСЯМ РУБРИКИ
                       пунктов  восстановлено
axis                                         
accuracy                    58       0.362069
communication_quality       69       0.260870
completeness               293       0.447099
reasoning                   28       0.571429
uncertainty                166       0.349398
urgency                     29       0.448276



M2. ВЫХОД ПАНЕЛИ ~ МИНИМУМ / СРЕДНЕЕ / МАКСИМУМ УЧАСТНИКОВ
Если значим только mx -- панель есть отбор лучшего; если mn тоже -- слабейший тянет вниз.
                          R2    n        mn          p mn      mean        p mean        mx          p mx
спецификация                                                                                             
y ~ mn              0.308758  228  0.415911  7.139019e-20       NaN           NaN       NaN           NaN
y ~ mean            0.421884  228       NaN           NaN  0.579743  1.039839e-28       NaN           NaN
y ~ mx              0.413233  228       NaN           NaN       NaN           NaN  0.580537  5.626468e-28
y ~ mn + mx         0.434948  228  0.154637  3.618740e-03       NaN           NaN  0.449769  1.729963e-11
y ~ mn + mean + mx  0.437694  228  0.014635  9.190409e-01  0.307160  2.967613e-01  0.285619  9.296582e-02



M3. «СЛАБЫМ БОЛЬШЕ»: НАИВНАЯ ОЦЕНКА ПРОТИВ ИНСТРУМЕНТАЛЬНОЙ
Инструмент снимает регрессию к середине: уровень берётся из независимых прогонов.
                                              n       rho             p                                    смещение
оценка                                                                                                             
наивная (уровень из того же замера)         228 -0.514220  8.652274e-17  регрессия к середине -> завышена по модулю
инструментальная (уровень из retest r1-r2)   59 -0.349386  6.681364e-03                          общий шум устранён
  наклон при инструментальном уровне         59 -0.403735  1.885780e-03                                         МНК



M4. СЛУЧАЙНЫЕ РОЛИ КАК ВТОРАЯ «ПОДДЕРЖКА»
                         noroles  randomroles  Δ от ролей             p
плечо                                                                  
baseline gpt            0.625811     0.648759    0.022948  5.192725e-02
baseline qwen           0.562228     0.595178    0.032950  8.455366e-05
baseline deepseekv4     0.640056     0.660308    0.020253  1.304272e-02
baseline deepseekv4pro  0.560091     0.599425    0.039334  3.335964e-04
baseline gemini         0.424720     0.524796    0.100076  2.700460e-15
baseline glm            0.587015     0.655186    0.068172  1.389856e-07
baseline kimi           0.590136     0.604794    0.014659  5.204681e-02
P-base Delphi           0.789946     0.813306    0.023360  2.793301e-03

  роли помогают тем сильнее, чем слабее модель: rho = -0.786, p = 0.0362 (по 7 моделям)

  НАКЛОН «прирост ОТ ПРОТОКОЛА ~ уровень участников» по дизайнам:
    noroles      -0.420 (p = 1.2e-17)
    randomroles  -0.453 (p = 1.2e-19)
    в